# Topic Analysis: 368 Topics Across Popularity Tiers

**Scope:** Topic-level analysis using BERTopic probabilities for all 368 topics. Statistical comparisons will use all three tiers (**Top / Middle / Trash**); pairwise contrasts (e.g., Top vs Trash) can be added as needed. Category-level statistics (taxonomy groups, Radway phases) are **excluded** here and will be handled in separate notebooks.

**Data sources (absolute paths):**
- Book features (wide): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_wide.parquet`
- Book features (long): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_long.parquet`
- Book topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet`
- Chapter topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/chapter_topic_probs.parquet`
- Topic lookup (REQUIRED): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/taxonomy_radway_eda/topic_lookup.parquet` - Required for topic labels (used instead of topic IDs in all statistical outputs)

**Outputs:** `results/stage10_correlation_analysis/topic_analysis_all_368/` (figures, tables)

## Roadmap (Top / Middle / Trash)

1) Setup & paths
2) Load data + integrity checks (prob sums per book/segment, n_topics=368)
3) **Merge labels from topic_lookup** (REQUIRED - labels used instead of topic IDs in all outputs)
4) Rating-tier prep: map rating_class → {top, middle, trash}; use all three tiers; optional pairwise views (Top vs Trash) if needed
5) Helper utilities: plotting helper, Cliff's delta, prevalence metrics
6) Topic health metrics (prevalence, mass, concentration) — using labels, not taxonomy columns
7) Topic-level distributions & summaries (Top vs Middle vs Trash): medians, means, effect sizes, FDR-corrected tests — **all outputs use labels**
8) Leaderboards & exports (tables + optional plot stubs) — **labeled by topic labels**
9) Optional: per-topic visualization hooks (violin/ECDF/box) for selected topic labels

In [4]:
# 1. Setup & imports
# NOTE: Always use venv for Python commands
# If running from terminal: source venv/bin/activate

from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Set plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Inline plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import warnings
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm.auto')

PROJECT_ROOT = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

✓ PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor


## Generate topic_lookup.parquet (if missing)

If `topic_lookup.parquet` is missing, run this command in terminal (with venv activated):

```bash
source venv/bin/activate
python src/stage10_correlation_analysis/data_preparation/01_data_validation_extraction.py \
    --output-dir results/stage10_correlation_analysis/data_preparation/taxonomy_radway_eda \
    --excluded-book-ids notebooks/07_analysis/statistical_analysis/excluded_book_ids.csv
```

This will generate `topic_lookup.parquet` with topic labels from Stage 08 LLM labeling.



In [5]:
# 2. Paths
DATA_PREP_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "data_preparation"
BOOK_FEATURES_DIR = DATA_PREP_DIR / "book_features"
TOPIC_PROBS_DIR = DATA_PREP_DIR / "topic_probabilities"
TAXONOMY_RADWAY_DIR = DATA_PREP_DIR / "taxonomy_radway_eda"

# Data file paths
BOOK_WIDE_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_wide.parquet"
BOOK_LONG_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_long.parquet"
BOOK_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "book_topic_probs.parquet"
CHAPTER_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "chapter_topic_probs.parquet"
TOPIC_LOOKUP_PATH = TAXONOMY_RADWAY_DIR / "topic_lookup.parquet"
GOODREADS_PATH = PROJECT_ROOT / "data" / "processed" / "goodreads.csv"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "topic_analysis_all_368"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data paths:")
print(f"  Book features (wide): {BOOK_WIDE_PATH}")
print(f"  Book features (long): {BOOK_LONG_PATH}")
print(f"  Book topic probs: {BOOK_TOPIC_PROBS_PATH}")
print(f"  Chapter topic probs: {CHAPTER_TOPIC_PROBS_PATH}")
print(f"  Topic lookup: {TOPIC_LOOKUP_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

Data paths:
  Book features (wide): /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_wide.parquet
  Book features (long): /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_long.parquet
  Book topic probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet
  Chapter topic probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/chapter_topic_probs.parquet
  Topic lookup: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/res

In [6]:
# 3. Load data
book_wide = pd.read_parquet(BOOK_WIDE_PATH)
book_long = pd.read_parquet(BOOK_LONG_PATH)
book_topic_probs = pd.read_parquet(BOOK_TOPIC_PROBS_PATH)
chapter_topic_probs = pd.read_parquet(CHAPTER_TOPIC_PROBS_PATH)

# Topic lookup is REQUIRED for labels (used instead of topic IDs in statistical analysis)
if not TOPIC_LOOKUP_PATH.exists():
    raise FileNotFoundError(
        f"topic_lookup.parquet is REQUIRED but not found at {TOPIC_LOOKUP_PATH}\n"
        f"Please run: python src/stage10_correlation_analysis/data_preparation/01_data_validation_extraction.py"
    )

topic_lookup = pd.read_parquet(TOPIC_LOOKUP_PATH)
print("✓ Loaded topic_lookup (REQUIRED for labels)")

# Verify topic_lookup has label column
if "label" not in topic_lookup.columns:
    raise ValueError("topic_lookup must have 'label' column. Check that Stage 08 labels were integrated.")

print("\nShapes:")
print(f"  book_wide: {book_wide.shape}")
print(f"  book_long: {book_long.shape}")
print(f"  book_topic_probs: {book_topic_probs.shape}")
print(f"  chapter_topic_probs: {chapter_topic_probs.shape}")
print(f"  topic_lookup: {topic_lookup.shape}")

# Merge labels into topic probabilities (for use in statistical analysis)
print("\nMerging labels from topic_lookup into book_topic_probs...")
book_topic_probs = book_topic_probs.merge(
    topic_lookup[["topic_id", "label"]].drop_duplicates("topic_id"),
    on="topic_id",
    how="left"
)

# Check label coverage
n_topics_with_labels = book_topic_probs["label"].notna().sum()
n_total = len(book_topic_probs)
print(f"  Topics with labels: {n_topics_with_labels:,} / {n_total:,} ({n_topics_with_labels/n_total*100:.1f}%)")
if book_topic_probs["label"].isna().any():
    missing_topics = book_topic_probs[book_topic_probs["label"].isna()]["topic_id"].unique()
    print(f"  ⚠️  {len(missing_topics)} topics missing labels: {sorted(missing_topics)[:10]}...")


✓ Loaded topic_lookup (REQUIRED for labels)

Shapes:
  book_wide: (92, 29)
  book_long: (2215, 22)
  book_topic_probs: (33856, 3)
  chapter_topic_probs: (1089280, 4)
  topic_lookup: (368, 20)

Merging labels from topic_lookup into book_topic_probs...
  Topics with labels: 33,856 / 33,856 (100.0%)


In [7]:
# 3b. Fix data issues: book_id type mismatch, load rating_class, check NaN probabilities

print("=" * 80)
print("Fixing data issues:")
print("=" * 80)

# 1. Fix book_id type mismatch: convert book_topic_probs book_id from string to int
print("\n1. Fixing book_id type mismatch...")
print(f"  book_wide['book_id'].dtype: {book_wide['book_id'].dtype}")
print(f"  book_topic_probs['book_id'].dtype: {book_topic_probs['book_id'].dtype}")

# Convert book_topic_probs book_id to numeric (matching book_wide)
book_topic_probs['book_id'] = pd.to_numeric(book_topic_probs['book_id'], errors='coerce')
print(f"  After conversion - book_topic_probs['book_id'].dtype: {book_topic_probs['book_id'].dtype}")

# Check for any conversion failures
failed_conversions = book_topic_probs['book_id'].isna().sum()
if failed_conversions > 0:
    print(f"  ⚠️  Warning: {failed_conversions} book_ids failed to convert to numeric")

# 2. Check NaN probabilities
print("\n2. Checking NaN probabilities...")
nan_probs = book_topic_probs['prob'].isna().sum()
total_probs = len(book_topic_probs)
print(f"  NaN probabilities: {nan_probs:,} / {total_probs:,} ({nan_probs/total_probs*100:.1f}%)")

if nan_probs > 0:
    print("  ⚠️  Warning: Found NaN probabilities. Checking which topics/books are affected...")
    nan_by_topic = book_topic_probs[book_topic_probs['prob'].isna()].groupby('topic_id').size().sort_values(ascending=False)
    nan_by_book = book_topic_probs[book_topic_probs['prob'].isna()].groupby('book_id').size().sort_values(ascending=False)
    print(f"  Topics with NaN: {len(nan_by_topic)} unique topics")
    print(f"  Books with NaN: {len(nan_by_book)} unique books")
    print(f"  Sample topics with NaN: {nan_by_topic.head(10).to_dict()}")
    
    # Drop rows with NaN probabilities for now (we'll investigate the root cause separately)
    print("  Dropping rows with NaN probabilities...")
    book_topic_probs = book_topic_probs.dropna(subset=['prob'])
    print(f"  After dropping NaN: {len(book_topic_probs):,} rows remaining")

# 3. Load goodreads.csv and create rating_class
print("\n3. Loading goodreads.csv and creating rating_class...")
if not GOODREADS_PATH.exists():
    raise FileNotFoundError(f"goodreads.csv not found at {GOODREADS_PATH}")

goodreads = pd.read_csv(GOODREADS_PATH)
print(f"  Loaded goodreads.csv: {goodreads.shape}")
print(f"  Columns: {list(goodreads.columns)}")

# Standardize book_id column name (could be ID, book_id, etc.)
if 'ID' in goodreads.columns:
    goodreads = goodreads.rename(columns={'ID': 'book_id'})
elif 'goodreads_book_id' in goodreads.columns:
    goodreads = goodreads.rename(columns={'goodreads_book_id': 'book_id'})

# Convert book_id to numeric to match book_wide
goodreads['book_id'] = pd.to_numeric(goodreads['book_id'], errors='coerce')

# Check if Score column exists (this is the rating)
if 'Score' not in goodreads.columns:
    print(f"  ⚠️  Warning: 'Score' column not found. Available columns: {list(goodreads.columns)}")
    # Try alternative names
    if 'rating_mean' in goodreads.columns:
        goodreads = goodreads.rename(columns={'rating_mean': 'Score'})
    elif 'avg_rating' in goodreads.columns:
        goodreads = goodreads.rename(columns={'avg_rating': 'Score'})
    else:
        raise KeyError(f"Could not find rating column. Available: {list(goodreads.columns)}")

# Create rating_class using quantiles (0.33, 0.66) as per methodology
print("  Creating rating_class based on Score quantiles...")
book_ratings = goodreads['Score'].dropna()
low_q, high_q = book_ratings.quantile([0.33, 0.66])
print(f"  Quantiles: {low_q:.3f} (33rd), {high_q:.3f} (66th)")

def assign_rating_class(score):
    if pd.isna(score):
        return None
    if score < low_q:
        return "bad"
    elif score <= high_q:
        return "mid"
    else:
        return "good"

goodreads['rating_class'] = goodreads['Score'].apply(assign_rating_class)
print(f"  Rating class distribution:")
print(goodreads['rating_class'].value_counts())

# 4. Merge rating_class into book_wide
print("\n4. Merging rating_class into book_wide...")
book_wide = book_wide.merge(
    goodreads[['book_id', 'rating_class', 'Score']],
    on='book_id',
    how='left'
)
print(f"  After merge: {book_wide.shape}")
print(f"  Books with rating_class: {book_wide['rating_class'].notna().sum()} / {len(book_wide)}")
print(f"  Rating class distribution in book_wide:")
print(book_wide['rating_class'].value_counts(dropna=False))

print("\n✓ Data fixes completed!")

Fixing data issues:

1. Fixing book_id type mismatch...
  book_wide['book_id'].dtype: Int64
  book_topic_probs['book_id'].dtype: object
  After conversion - book_topic_probs['book_id'].dtype: int64

2. Checking NaN probabilities...
  NaN probabilities: 0 / 33,856 (0.0%)

3. Loading goodreads.csv and creating rating_class...
  Loaded goodreads.csv: (97, 14)
  Columns: ['ID', 'Author', 'Title', 'URL', 'SeriesName', 'Summary', 'Genres', 'Score', 'RatingsCount', 'ReviewsCount', 'Pages', 'PublishedDate', 'Popularity_ReadingNow', 'Popularity_Wishlisted']
  Creating rating_class based on Score quantiles...
  Quantiles: 3.917 (33rd), 4.074 (66th)
  Rating class distribution:
rating_class
good    33
mid     32
bad     32
Name: count, dtype: int64

4. Merging rating_class into book_wide...
  After merge: (92, 31)
  Books with rating_class: 92 / 92
  Rating class distribution in book_wide:
rating_class
mid     32
good    30
bad     30
Name: count, dtype: int64

✓ Data fixes completed!


In [8]:
# 4. Integrity checks
n_topics_expected = 368

print("=" * 80)
print("Integrity checks:")
print("=" * 80)

# Check topics count
print("Unique topics (book):", book_topic_probs["topic_id"].nunique())
print("Unique topics (chapter):", chapter_topic_probs["topic_id"].nunique())

# Probability sums per book
book_sums = book_topic_probs.groupby("book_id")["prob"].sum()
print("Book prob sums — min/max:", book_sums.min(), book_sums.max())

# Probability sums per (book, chapter)
if {"book_id", "chapter_id"}.issubset(chapter_topic_probs.columns):
    chapter_sums = chapter_topic_probs.groupby(["book_id", "chapter_id"])["prob"].sum()
    print("Chapter prob sums — min/max:", chapter_sums.min(), chapter_sums.max())

# Basic cohort overlap (should now work after fixing book_id types)
book_ids_probs = set(book_topic_probs["book_id"].dropna().unique())
book_ids_wide = set(book_wide["book_id"].dropna().unique())
overlap = book_ids_probs & book_ids_wide
print("Book IDs overlap (topic_probs ∩ wide):", len(overlap), "of", len(book_ids_probs))
if len(overlap) == 0:
    print("  ⚠️  WARNING: No overlap! Check book_id formats:")
    print(f"  book_ids_probs sample: {sorted(list(book_ids_probs))[:5]}")
    print(f"  book_ids_wide sample: {sorted(list(book_ids_wide))[:5]}")
    print(f"  book_ids_probs dtype: {book_topic_probs['book_id'].dtype}")
    print(f"  book_ids_wide dtype: {book_wide['book_id'].dtype}")

# Rating class availability (should now be present after merge)
if "rating_class" in book_wide.columns:
    print("✓ Rating classes present:", sorted(book_wide["rating_class"].dropna().unique()))
    print("  Rating class distribution:")
    print(book_wide["rating_class"].value_counts(dropna=False))
else:
    print("⚠️ rating_class column missing in book_wide")

Integrity checks:
Unique topics (book): 368
Unique topics (chapter): 368
Book prob sums — min/max: 0.6201516892595613 0.7312169573083945
Chapter prob sums — min/max: 0.3114608353220605 0.9285893186535736
Book IDs overlap (topic_probs ∩ wide): 92 of 92
✓ Rating classes present: ['bad', 'good', 'mid']
  Rating class distribution:
rating_class
mid     32
good    30
bad     30
Name: count, dtype: int64


In [9]:
# 6. Rating-tier prep (Top / Middle / Trash)
# rating_class should already be in book_wide from the previous cell
print("=" * 80)
print("Rating-tier preparation:")
print("=" * 80)

# Expect rating_class in {"good","mid","bad"}; map to desired labels
rating_map = {
    "good": "top",
    "bad": "trash",
    "mid": "middle",
    "top": "top",
    "trash": "trash",
    "middle": "middle",
}

if "rating_class" not in book_wide.columns:
    print("⚠️ rating_class not found in book_wide.columns")
    print("Available columns:", list(book_wide.columns))
    raise KeyError("rating_class not found in book_wide; please check the data loading and merge steps")

# Check what values we have
print("rating_class values in book_wide:", sorted(book_wide["rating_class"].dropna().unique()))

book_wide = book_wide.copy()
book_wide["rating_tier"] = book_wide["rating_class"].map(rating_map)

# Check for unmapped values
unmapped = book_wide[book_wide["rating_tier"].isna() & book_wide["rating_class"].notna()]
if len(unmapped) > 0:
    print(f"⚠️  Warning: {len(unmapped)} books with unmapped rating_class values:")
    print(unmapped["rating_class"].value_counts())

# Merge rating_tier onto topic probabilities
print("\nMerging rating_tier onto topic probabilities...")
book_topic_probs = book_topic_probs.merge(
    book_wide[["book_id", "rating_tier"]], on="book_id", how="left"
)

# Check merge success
merged_count = book_topic_probs["rating_tier"].notna().sum()
total_count = len(book_topic_probs)
print(f"  Merged rating_tier: {merged_count:,} / {total_count:,} ({merged_count/total_count*100:.1f}%)")

# Use all three tiers by default
book_topic_probs_all = book_topic_probs.copy()

# Optional helper: Top vs Trash subset for pairwise contrasts if needed later
book_topic_probs_top_trash = book_topic_probs_all[book_topic_probs_all["rating_tier"].isin(["top", "trash"])].copy()

print("\nBooks per tier (all):")
print(book_topic_probs_all.groupby("rating_tier")["book_id"].nunique())
print("\nBooks per tier (Top/Trash subset, optional):")
print(book_topic_probs_top_trash.groupby("rating_tier")["book_id"].nunique())

Rating-tier preparation:
rating_class values in book_wide: ['bad', 'good', 'mid']

Merging rating_tier onto topic probabilities...
  Merged rating_tier: 33,856 / 33,856 (100.0%)

Books per tier (all):
rating_tier
middle    32
top       30
trash     30
Name: book_id, dtype: int64

Books per tier (Top/Trash subset, optional):
rating_tier
top      30
trash    30
Name: book_id, dtype: int64


In [10]:
# 7. Helper utilities

def show_plotly_fig(fig, save_html=True, output_dir=FIG_DIR):
    """Display Plotly figure; optionally save to HTML."""
    try:
        fig.show()
    except (ValueError, ImportError):
        pass
    if save_html and output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        html_path = output_dir / f"plot_{hash(str(fig.layout.title.text if fig.layout.title else 'figure'))}.html"
        fig.write_html(str(html_path))
        print(f"Saved interactive plot to: {html_path}")


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta effect size."""
    x = np.asarray(x)
    y = np.asarray(y)
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (len(x) * len(y))


def compute_topic_health(df: pd.DataFrame, threshold: float = 0.001) -> pd.DataFrame:
    """Compute prevalence/mass/concentration per topic using LABELS (not topic_id)."""
    # Group by label instead of topic_id for outputs
    prevalence = (df["prob"] > threshold).groupby(df["label"]).mean().rename("prevalence")
    mass = df.groupby("label")["prob"].mean().rename("mass")
    # simple concentration proxy: max share / mean share per topic
    concentration = (
        df.groupby("label")["prob"].max() / df.groupby("label")["prob"].mean().replace(0, np.nan)
    ).rename("concentration_ratio")
    # Also keep topic_id for reference
    topic_id_map = df.groupby("label")["topic_id"].first().rename("topic_id")
    out = pd.concat([prevalence, mass, concentration, topic_id_map], axis=1).reset_index()
    return out